# **Train YOLOv26-Pose on AwA2 dataset**

## ***Create YOLO dataset***

In [1]:
!invidia-smi

/bin/bash: line 1: invidia-smi: command not found


In [2]:
!pip install ultralytics onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00


In [3]:
print(100*'#')
!pip show ultralytics
print(100*'-')
!pip show onnx
print(100*'-')
!pip show onnxruntime
print(100*'#')

####################################################################################################
Name: ultralytics
Version: 8.4.5
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: matplotlib, numpy, opencv-python, pillow, polars, psutil, pyyaml, requests, scipy, torch, torchvision, ultralytics-thop
Required-by: 
----------------------------------------------------------------------------------------------------
Name: onnx
Version: 1.20.1
Summary: Open Neural Network Exchange
Home-page: https://onnx.ai/
Author: 
Author-email: ONNX Contributors <onnx-technical-discuss@lists.lfaidata.foundation>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: ml_dtypes, numpy

### ***Imports***

In [4]:
import os
import yaml
import base64
import io

import pandas as pd
import numpy as np
import torch

from ultralytics import YOLO

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### ***Konfiguration***

In [5]:
DATASET_ROOT = os.path.join('/kaggle','input','awa2-dataset','AwA2_pose','lite_dataset')  # folder with three .parquet
OUTPUT_ROOT  = os.path.join('/kaggle','working','awa2_yolo_pose')    # where to save the final dataset
IMG_SUBFOLDER = "images"
LBL_SUBFOLDER = "labels"

In [6]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER), exist_ok=True)

#### ***Creating folders***

In [7]:
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER,split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER,split), exist_ok=True)

#### ***Columns with keys***

In [8]:
keypoint_columns = [
    'nose','belly_bottom',
    'right_eye','left_eye',
    'right_antler_base','left_antler_base',
    'right_antler_end','left_antler_end',
    'right_earend','left_earend',
    'right_earbase','left_earbase',
    'mouth_end_right','mouth_end_left',
    'lower_jaw','upper_jaw',
    'body_middle_right','body_middle_left',
    'front_right_knee','front_left_knee',
    'back_right_knee','back_left_knee',
    'front_right_paw','front_left_paw',
    'back_right_paw','back_left_paw',
    'front_right_thai','front_left_thai',
    'back_right_thai','back_left_thai',
    'throat_base','throat_end',
    'tail_base','tail_end',
    'neck_base','neck_end',
    'back_base','back_middle','back_end',
]

In [9]:
NUM_KPTS = len(keypoint_columns)                  # z.B. 29–39
print(f"{NUM_KPTS} keypoints used")

39 keypoints used


#### ***Data retrieval***

In [10]:
splits = {
    "train": pd.read_parquet(os.path.join(DATASET_ROOT,"train_lite.parquet")),
    "val":   pd.read_parquet(os.path.join(DATASET_ROOT,"validation_lite.parquet")),
    "test":  pd.read_parquet(os.path.join(DATASET_ROOT,"test_lite.parquet")),
}

####  ***Label encoder for classes (0–33)***

In [11]:
all_classes = pd.concat([df["name_class"] for df in splits.values()]).unique()
le = LabelEncoder().fit(all_classes)

In [12]:
print("Number of classes:", len(all_classes))
print("Example mapping:", dict(zip(all_classes[:5], le.transform(all_classes[:5]))))

Number of classes: 34
Example mapping: {'polar+bear': np.int64(23), 'ox': np.int64(20), 'sheep': np.int64(27), 'squirrel': np.int64(29), 'antelope': np.int64(0)}


### ***Main conversion***

In [13]:
for split_name, df in splits.items():
    print(f"\nProcessing split: {split_name} ({len(df)} images)")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # 1. Decode the image
        try:
            img_bytes = base64.b64decode(row["image_base64s"])
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception as e:
            print(f"Error decoding image {row.get('name_file', idx)}: {e}")
            continue

        w, h = row["image_width"], row["image_height"]
        assert img.size == (w, h), "The dimensions don't fit!"

        # Save image
        img_name = f"{row.get('name_file', f'img_{idx:06d}')}.jpg"
        img_path = os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER,split_name,img_name)
        img.save(img_path)

        # 2. Bounding box – [x1,y1,x2,y2] → YOLO normalized format
        if "bbox" not in row or len(row["bbox"]) != 4:
            continue  # skip without bbox
        x1, y1, x2, y2 = row["bbox"]
        xc = (x1 + x2) / 2 / w
        yc = (y1 + y2) / 2 / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        # 3. Key points + visibility
        kpts_norm = []
        for kp_col in keypoint_columns:
            kp_value = row.get(kp_col, [-1, -1])
        
            if kp_value is None or (isinstance(kp_value, float) and np.isnan(kp_value)):
                kp = [-1, -1]
            else:
                kp = kp_value
        
            # Format check
            if not isinstance(kp, (list, tuple, np.ndarray)) or len(kp) != 2:
                kpts_norm.extend([0.0, 0.0, 0])
                continue
        
            # Convert to a regular sheet (just in case)
            kp = list(kp)
        
            x, y = kp
        
            if x < 0 or y < 0:
                x_norm = 0.0
                y_norm = 0.0
                vis = 0
            else:
                x_norm = float(np.clip(x / w, 0.0, 1.0))
                y_norm = float(np.clip(y / h, 0.0, 1.0))
                vis = 2
        
            kpts_norm.extend([x_norm, y_norm, vis])

        # 4. class id
        class_id = le.transform([row["name_class"]])[0]

        #5. .txt writing
        label_path = os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER,split_name, img_name.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            line = f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
            for val in kpts_norm:
                if isinstance(val, float):
                    line += f" {val:.6f}"
                else:
                    line += f" {val}"
            f.write(line + "\n")

print("\nDone! Dataset ready in:", OUTPUT_ROOT)


Processing split: train (9423 images)


100%|██████████| 9423/9423 [01:17<00:00, 122.09it/s]



Processing split: val (524 images)


100%|██████████| 524/524 [00:04<00:00, 124.78it/s]



Processing split: test (524 images)


100%|██████████| 524/524 [00:04<00:00, 118.87it/s]


Done! Dataset ready in: /kaggle/working/awa2_yolo_pose


### ***We will create the file awa2-pose-39kpt.yaml***

In [14]:
df = pd.read_parquet(os.path.join(DATASET_ROOT,'train_lite.parquet'))
class_names = sorted(df["name_class"].unique().tolist())
print(len(class_names))          # should be 34
print(class_names)

34
['antelope', 'bobcat', 'buffalo', 'chihuahua', 'collie', 'cow', 'dalmatian', 'deer', 'elephant', 'fox', 'german+shepherd', 'giant+panda', 'giraffe', 'grizzly+bear', 'hippopotamus', 'horse', 'leopard', 'lion', 'moose', 'otter', 'ox', 'persian+cat', 'pig', 'polar+bear', 'rabbit', 'raccoon', 'rhinoceros', 'sheep', 'siamese+cat', 'squirrel', 'tiger', 'weasel', 'wolf', 'zebra']


### ***YAML file content***

In [15]:
dataset_config = {
    "path": OUTPUT_ROOT,          # real absolute or relative path
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",       # optional
    
    "nc": 34,
    "names": class_names,
    
    "kpt_shape": [39, 3],
    
    "flip_idx": [0, 1, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 13, 12, 14, 15, 17, 16, 19, 18, 21, 20, 23, 22, 25, 24, 27, 26, 29, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38],
    
    # augmentation parameters (optional - good for animals)
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.2,
    "scale": 0.5,
    "shear": 3.0,
    "perspective": 0.0,
    "flipud": 0.4,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.2,
}

### ***Save to file***

In [16]:
yaml_path = os.path.join('/kaggle','working','awa2-pose-39kpt.yaml')

In [17]:
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(
        dataset_config,
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False
    )

print(f"File created: {yaml_path}")

File created: /kaggle/working/awa2-pose-39kpt.yaml


## ***Training***

In [18]:
print("GPU available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device:", torch.cuda.get_device_name(0))

GPU available: True
Device count: 1
Current device: Tesla P100-PCIE-16GB


#### ***Loading a pre-trained model (best start for a custom pose)***

In [19]:
model = YOLO("yolo26n-pose.pt")   # nano – fast startup, low memory
# Alternatives: "yolo26n-pose.pt", "yolo26m-pose.pt" (if you have more GPU/VRAM)

#### ***Training***

In [20]:
results = model.train(
    data      = yaml_path,   # path to yaml
    epochs    = 80,                  # 60–150 depending on data and GPU time
    imgsz     = 640,
    batch     = 32,                  # adjust 16–48 according to VRAM (RTX 4000+ → 32–48)
    device    = 0,                   # or [0,1] for multi-GPU
    workers   = 8,
    patience  = 35,                  # early stopping
    #cache     = "disk",              # faster loading (if you have disk space)
    project   = os.path.join('/kaggle','working','runs'),
    name      = "yolo26n-awa2-pose-39kpt",
    
    # augmentation – animals need stronger variability
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    degrees   = 12.0,
    translate = 0.25,
    scale     = 0.6,
    shear     = 4.0,
    perspective = 0.0,
    flipud    = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.15,
    
    # other useful
    amp       = True,                # automatic mixed precision
    single_cls = False,
    plots     = True,                # stores confusion matrix, PR curves, etc.
)

Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/awa2-pose-39kpt.yaml, degrees=12.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n-awa2-pose-39kpt, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

>Po tréninku se nejlepší model uloží do
>
```cmd
/kaggle/working/runs/yolo12n-awa2-pose-39kpt/weights/best.pt
```

## ***Validation on val set***

#### ***Load the best model from training***

In [21]:
best_model = YOLO(os.path.join('/kaggle','working','runs','yolo26n-awa2-pose-39kpt','weights','best.pt'))

#### ***Validation***

In [22]:
metrics = best_model.val(
    data   = yaml_path,
    split  = "val",
    imgsz  = 640,
    batch  = 16,
    device = 0,
    plots  = True,          # confusion matrix, PR curves, etc.
    save_json = True,       # for later analysis
)

Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO26n-pose summary (fused): 132 layers, 4,264,347 parameters, 0 gradients, 13.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2744.4±526.4 MB/s, size: 159.3 KB)
val: Scanning /kaggle/working/awa2_yolo_pose/labels/val.cache... 524 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 524/524 219.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 5.9it/s 5.6s
                   all        524        524      0.835      0.866      0.917      0.736       0.58      0.537        0.5      0.119
              antelope         19         19      0.809      0.894       0.94      0.783      0.807      0.737      0.791       0.21
                bobcat         17         17       0.94      0.921      0.972      0.798      0.705      0.647      0.588      0.192
               buf

In [23]:
print(metrics)

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7da116893cb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.02402

>Key pose metrics:
>
>- **Box mAP50**, **Box mAP50-95**
>- **Pose mAP50**, **Pose mAP50-95** ← keypoints main quality indicator
>- **OKS** (Object Keypoint Similarity) is used internally

## ***Inference (prediction) on test set + visualization***

#### ***Test set inference + save results***

In [24]:
test_results = best_model.predict(
    source       = os.path.join('/kaggle','working','awa2_yolo_pose','images','test'),   # folder with test images
    save         = True,             # saves images with predictions
    save_txt     = True,             # saves .txt with detections (YOLO format)
    save_conf    = True,
    conf         = 0.25,
    iou          = 0.45,
    imgsz        = 640,
    project      = os.path.join('/kaggle','working','runs','predict_test'),
    name         = "yolo26n-awa2-test",
    exist_ok     = True,
)


image 1/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10036.jpg: 640x640 1 antelope, 10.9ms
image 2/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10066.jpg: 512x640 2 antelopes, 67.1ms
image 3/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10075.jpg: 640x480 1 antelope, 1 deer, 67.5ms
image 4/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10178.jpg: 640x448 1 antelope, 65.7ms
image 5/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10342.jpg: 544x640 2 antelopes, 65.6ms
image 6/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10348.jpg: 448x640 1 antelope, 65.0ms
image 7/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10393.jpg: 640x448 (no detections), 10.6ms
image 8/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10404.jpg: 512x640 1 antelope, 10.2ms
image 9/524 /kaggle/working/awa2_yolo_pose/images/test/antelope_10466.jpg: 480x640 1 antelope, 1 deer, 63.6ms
image 10/524 /kaggle/working/awa2_yolo_pose/images/test/an

In [25]:
print(f"Number of processed images: {len(test_results)}")

Number of processed images: 524


You can find the output in
```cmd
/kaggle/working/runs/predict_test/yolo12n-awa2-test/
```
→ images with drawn bounding boxes + keypoints

## ***Export do ONNX***

### ***Export the best model to ONNX***

In [26]:
success = best_model.export(
    format   = "onnx",
    imgsz    = 640,
    dynamic  = False,           # True = dynamic dimensions (sometimes slower)
    simplify = True,            # shrinks the model, often faster inference
    opset    = 17,              # safe compromise (18+ sometimes problems)
)

Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/yolo26n-awa2-pose-39kpt/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 123) (12.7 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 14 packages in 204ms
Prepared 2 packages in 2.64s
Installed 2 packages in 112ms
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.82

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 17...


/usr/local/lib/python3.12/dist-packages/torch/onnx/symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 5.4s, saved as '/kaggle/working/runs/yolo26n-awa2-pose-39kpt/weights/best.onnx' (16.7 MB)

Export complete (5.9s)
Results saved to /kaggle/working/runs/yolo26n-awa2-pose-39kpt/weights
Predict:         yolo predict task=pose model=/kaggle/working/runs/yolo26n-awa2-pose-39kpt/weights/best.onnx imgsz=640 
Validate:        yolo val task=pose model=/kaggle/working/runs/yolo26n-awa2-pose-39kpt/weights/best.onnx imgsz=640 data=/kaggle/working/awa2-pose-39kpt.yaml  
Visualize:       https://netron.app


In [27]:
print("ONNX export:", "successful" if success else "failed")

ONNX export: successful


Output: /kaggle/working/runs/yolo12n-awa2-pose-39kpt/weights/best.onnx

***Optional advanced export***

```python
# smaller model + half precision (FP16)
best_model.export(format="onnx", half=True, imgsz=640, simplify=True)

# dynamic dimensions (batch=1, height/width variable)
best_model.export(format="onnx", dynamic=True, imgsz=[640, 640])
```